**Install & Imports data **

In [ ]:
!pip install gradio tensorflow scikit-learn -q

import os
import re
import gradio as gr
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

**Clean Text + Load Dataset**

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = text.lower().strip()
    return text


df_all = pd.read_csv(
    "fake_reviews_dataset.csv",
    on_bad_lines='skip',
    engine='python'
)

df = df_all.sample(n=8000, random_state=42)

X = df["text_"].astype(str).apply(clean_text)
y = df["label"]

**TFIDF + Train/Test Split**

In [ ]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

accuracies = {}
trained_models = {}

**SVM Algo**

In [ ]:
svm_model = LinearSVC()
svm_model.fit(X_train, y_train)

pred = svm_model.predict(X_test)
acc = accuracy_score(y_test, pred)

accuracies["SVM"] = acc

print("================================")
print("SVM Accuracy:")
print(f"{acc*100:.2f}%")
print("================================")

SVM Accuracy:
90.62%


**Random forest Algo**

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

pred = rf_model.predict(X_test)
acc = accuracy_score(y_test, pred)

accuracies["Random Forest"] = acc

print("================================")
print("Random Forest Accuracy:")
print(f"{acc*100:.2f}%")
print("================================")

Random Forest Accuracy:
86.50%


**Naive Algo**

In [ ]:
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train, y_train)

pred = nb_model.predict(X_test)
acc = accuracy_score(y_test, pred)

accuracies["Naive Bayes"] = acc

print("================================")
print("Naive Bayes Accuracy:")
print(f"{acc*100:.2f}%")
print("================================")

Naive Bayes Accuracy:
88.88%


**Logistic Algo**

In [ ]:
lr_model = LogisticRegression(max_iter=200)
lr_model.fit(X_train, y_train)

pred = lr_model.predict(X_test)
acc = accuracy_score(y_test, pred)

accuracies["Logistic Regression"] = acc

print("================================")
print("Logistic Regression Accuracy:")
print(f"{acc*100:.2f}%")
print("================================")

Logistic Regression Accuracy:
89.31%


**LSTM Algo**

In [ ]:
max_words = 10000
max_len = 200

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X)

X_seq = tokenizer.texts_to_sequences(X)
X_pad = pad_sequences(X_seq, maxlen=max_len)

y_numeric = y.map({"OR": 0, "CG": 1})

X_train_lstm, X_test_lstm, y_train_lstm, y_test_lstm = train_test_split(
    X_pad, y_numeric, test_size=0.2, random_state=42
)

if os.path.exists("lstm_model.h5"):
    lstm_model = load_model("lstm_model.h5")
    print("Loaded LSTM Model")

else:
    lstm_model = Sequential([
        Embedding(input_dim=max_words, output_dim=128),
        Bidirectional(LSTM(64)),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])

    lstm_model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    lstm_model.fit(
        X_train_lstm,
        y_train_lstm,
        epochs=3,
        batch_size=64,
        validation_split=0.1
    )

    lstm_model.save("lstm_model.h5")

loss, acc_lstm = lstm_model.evaluate(X_test_lstm, y_test_lstm)

print("================================")
print("LSTM Accuracy:")
print(f"{acc_lstm*100:.2f}%")
print("================================")

Loaded LSTM Model
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 88ms/step - accuracy: 0.8863 - loss: 0.2782
LSTM Accuracy:
88.63%


**Prediction Function**

In [ ]:
def predict_review(text):

    if not text.strip():
        return ["Input is empty!"] * 5

    cleaned = clean_text(text)

    results = []

    vec = tfidf.transform([cleaned])

    for name, model in {
        "SVM": svm_model,
        "Random Forest": rf_model,
        "Naive Bayes": nb_model,
        "Logistic Regression": lr_model
    }.items():

        pred = model.predict(vec)[0]
        label = " FAKE" if (pred == "CG" or pred == 1) else " REAL"

        results.append(label)

    seq = tokenizer.texts_to_sequences([cleaned])
    pad = pad_sequences(seq, maxlen=max_len)

    pred_lstm = lstm_model.predict(pad)[0][0]
    label_lstm = " FAKE" if pred_lstm >= 0.5 else " REAL"

    results.append(label_lstm)

    return results

**GUI**

In [ ]:
with gr.Blocks(title="Fake Review Detection System") as demo:

    gr.Markdown("#  Fake Review Detection System")

    user_input = gr.Textbox(
        label="Enter Review",
        lines=5,
        placeholder="Write review here..."
    )

    btn = gr.Button("Predict")

    out1 = gr.Textbox(label="SVM")
    out2 = gr.Textbox(label="Random Forest")
    out3 = gr.Textbox(label="Naive Bayes")
    out4 = gr.Textbox(label="Logistic Regression")
    out5 = gr.Textbox(label="LSTM")

    btn.click(
        fn=predict_review,
        inputs=user_input,
        outputs=[out1, out2, out3, out4, out5]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b0db41d1f10202ed55.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**الكود كله **

In [ ]:
!pip install gradio tensorflow -q

import os
import re
import gradio as gr
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from tensorflow.keras.models import Sequential, load_model

from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Dense,
    Bidirectional,
    Dropout
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

def clean_text(text):

    text = str(text)

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    text = text.lower().strip()

    return text

try:

    df_all = pd.read_csv(
        "fake_reviews_dataset.csv",
        on_bad_lines='skip',
        engine='python'
    )

    df = df_all.sample(
        n=8000,
        random_state=42
    )

    print(f" Dataset Loaded Successfully: {len(df)} samples")

except Exception as e:

    print(f" Error Loading Dataset: {e}")

X = df["text_"].astype(str).apply(clean_text)

y = df["label"]

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

X_tfidf = tfidf.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

models_list = {

    "SVM": LinearSVC(C=1.0),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    "Naive Bayes": MultinomialNB(alpha=0.1),

    "Logistic Regression": LogisticRegression(
        max_iter=200
    )
}

accuracies = {}
trained_models = {}

print("\n==============================")
print("Training Traditional Models")
print("==============================")

for name, model in models_list.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    acc = accuracy_score(y_test, predictions)

    accuracies[name] = f"{acc * 100:.2f}%"

    trained_models[name] = model

    print(f" {name}: {accuracies[name]}")

max_words = 10000
max_len = 200

tokenizer = Tokenizer(num_words=max_words)

tokenizer.fit_on_texts(X)

X_sequences = tokenizer.texts_to_sequences(X)

X_padded = pad_sequences(
    X_sequences,
    maxlen=max_len
)

y_numeric = y.map({
    "OR": 0,
    "CG": 1
})

X_train_lstm, X_test_lstm, y_train_lstm, y_test_lstm = train_test_split(
    X_padded,
    y_numeric,
    test_size=0.2,
    random_state=42
)

if os.path.exists("lstm_model.h5"):

    lstm_model = load_model("lstm_model.h5")

    print("\n Saved LSTM Model Loaded Successfully")

else:

    print("\n==============================")
    print("Training LSTM Model")
    print("==============================")

    lstm_model = Sequential([

        Embedding(
            input_dim=max_words,
            output_dim=128
        ),

        Bidirectional(
            LSTM(64)
        ),

        Dropout(0.5),

        Dense(1, activation='sigmoid')
    ])

    lstm_model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    lstm_model.fit(
        X_train_lstm,
        y_train_lstm,
        epochs=3,
        batch_size=64,
        validation_split=0.1
    )

    lstm_model.save("lstm_model.h5")

    print(" LSTM Model Saved")

loss, acc = lstm_model.evaluate(
    X_test_lstm,
    y_test_lstm
)

accuracies["LSTM"] = f"{acc * 100:.2f}%"

print(f" LSTM Accuracy: {accuracies['LSTM']}")

def predict_review(text):

    if not text.strip():

        return ["Input is empty!"] * 5

    cleaned_input = clean_text(text)

    results = []

    vec_input = tfidf.transform([cleaned_input])

    for name in [

        "SVM",
        "Random Forest",
        "Naive Bayes",
        "Logistic Regression"
    ]:

        pred = trained_models[name].predict(vec_input)[0]
        label = (
            " FAKE"
            if (pred == "CG" or pred == 1)
            else " REAL"
        )

        results.append(
            f"{label} | Accuracy: {accuracies[name]}"
        )

    seq = tokenizer.texts_to_sequences([cleaned_input])

    padded = pad_sequences(
        seq,
        maxlen=max_len
    )

    lstm_pred = lstm_model.predict(padded)[0][0]

    lstm_label = (
        " FAKE"
        if lstm_pred >= 0.5
        else " REAL"
    )

    results.append(
        f"{lstm_label} | Accuracy: {accuracies['LSTM']}"
    )

    return results

with gr.Blocks(title="Fake Review Identification") as demo:

    gr.Markdown(
        "#  Fake Product Review Identification System"
    )

    gr.Markdown(
        "### Academic Project | Spring 2025–2026"
    )

    with gr.Row():

        user_input = gr.Textbox(
            label="Input Review",
            placeholder="Paste your review here...",
            lines=5
        )

    process_btn = gr.Button(
        "Process",
        variant="primary"
    )

    gr.Markdown(
        "## Results of 5 Algorithms"
    )

    with gr.Column():

        out1 = gr.Textbox(
            label="Algorithm 1 - SVM"
        )

        out2 = gr.Textbox(
            label="Algorithm 2 - Random Forest"
        )

        out3 = gr.Textbox(
            label="Algorithm 3 - Naive Bayes"
        )

        out4 = gr.Textbox(
            label="Algorithm 4 - Logistic Regression"
        )

        out5 = gr.Textbox(
            label="Algorithm 5 - LSTM"
        )

    process_btn.click(
        fn=predict_review,
        inputs=user_input,
        outputs=[
            out1,
            out2,
            out3,
            out4,
            out5
        ]
    )

demo.launch(share=True)

 Dataset Loaded Successfully: 8000 samples

Training Traditional Models
 SVM: 90.62%
 Random Forest: 86.50%
 Naive Bayes: 88.88%
 Logistic Regression: 89.31%

Training LSTM Model
Epoch 1/3
90/90 ━━━━━━━━━━━━━━━━━━━━ 41s 409ms/step - accuracy: 0.7479 - loss: 0.5115 - val_accuracy: 0.8719 - val_loss: 0.3123
Epoch 2/3
90/90 ━━━━━━━━━━━━━━━━━━━━ 37s 405ms/step - accuracy: 0.9104 - loss: 0.2420 - val_accuracy: 0.9031 - val_loss: 0.2521
Epoch 3/3
90/90 ━━━━━━━━━━━━━━━━━━━━ 38s 419ms/step - accuracy: 0.9533 - loss: 0.1388 - val_accuracy: 0.9031 - val_loss: 0.2450


 LSTM Model Saved
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 58ms/step - accuracy: 0.9038 - loss: 0.2511
 LSTM Accuracy: 90.38%
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8ee5b45d15f3f93fa7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
